# DINOv3 零样本分割 - CPU/GPU 测试版

本 notebook 演示如何使用 DINOv3 + dino.txt 进行开放词汇的语义分割。

## 权重说明

| 权重文件 | 用途 | 状态 |
|---------|------|------|
| dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth | ViT-L Backbone | 本地加载 |
| dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth | Text Encoder | 本地加载 |

## 1. 环境配置

In [ ]:
import dataclasses
import math
import warnings
from typing import Callable
import os
import sys

import numpy as np
import PIL.Image
import torch
import torch.nn.functional as F
import torchvision.transforms as TVT
import torchvision.transforms.functional as TVTF
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

# warnings.filterwarnings('ignore')

# 配置 matplotlib 支持中文
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 设备选择：自动检测 GPU，否则使用 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch 版本: {torch.__version__}')
print(f'使用设备: {device}')

## 2. 加载 DINOv3 + dino.txt 模型

In [ ]:
# 配置路径 - 请根据你的实际路径修改
import sys
import gc
REPO_DIR = r'd:\AI\Git\dinov3.git'
sys.path.append(REPO_DIR)

# 权重路径 - 请根据你的实际路径修改
BACKBONE_WEIGHTS_PATH = r'd:\AI\Git\dinov3.git\notebooks\dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
DINOTXT_WEIGHTS_PATH = r'd:\AI\Git\dinov3.git\notebooks\dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth'
BPE_PATH = r'd:\AI\Git\dinov3.git\notebooks\bpe_simple_vocab_16e6.txt.gz'

print('正在加载 DINOv3 + dino.txt 模型...')
print(f'Backbone: {BACKBONE_WEIGHTS_PATH}')
print(f'DINO.txt: {DINOTXT_WEIGHTS_PATH}')

from dinov3.eval.text.dinotxt_model import DINOTxt, DINOTxtConfig
from dinov3.eval.text.text_transformer import TextTransformer
from dinov3.eval.text.tokenizer import Tokenizer
from dinov3.hub.backbones import dinov3_vitl16  # 使用 vitl16 匹配权重

# 配置
dinotxt_config = DINOTxtConfig(
    embed_dim=2048,
    vision_model_freeze_backbone=True,
    vision_model_train_img_size=224,
    vision_model_use_class_token=True,
    vision_model_use_patch_tokens=True,
    vision_model_num_head_blocks=2,
    vision_model_head_blocks_drop_path=0.3,
    vision_model_use_linear_projection=False,
    vision_model_patch_tokens_pooler_type='mean',
    vision_model_patch_token_layer=1,
    text_model_freeze_backbone=False,
    text_model_num_head_blocks=0,
    text_model_head_blocks_is_causal=False,
    text_model_head_blocks_drop_prob=0.0,
    text_model_tokens_pooler_type='argmax',
    text_model_use_linear_projection=True,
    init_logit_scale=math.log(1 / 0.07),
    init_logit_bias=None,
    freeze_logit_scale=False,
)

# 加载 backbone（使用内存映射减少内存占用）
print('加载 Backbone 权重...')
vision_backbone = dinov3_vitl16(pretrained=False)  # 使用 vitl16

# 使用 mmap 加载大文件，减少内存占用
try:
    backbone_state_dict = torch.load(BACKBONE_WEIGHTS_PATH, map_location='cpu', weights_only=True, mmap=True)
except TypeError:
    # 旧版 PyTorch 不支持 mmap 参数
    backbone_state_dict = torch.load(BACKBONE_WEIGHTS_PATH, map_location='cpu', weights_only=True)

vision_backbone.load_state_dict(backbone_state_dict, strict=True)
print('Backbone 权重加载完成')

# 删除 state_dict 释放内存
del backbone_state_dict
gc.collect()

# 创建 text backbone
print('创建 Text Backbone...')
text_backbone = TextTransformer(
    context_length=77,
    vocab_size=49408,
    dim=1280,
    num_heads=20,
    num_layers=24,
    ffn_ratio=4,
    is_causal=True,
    ls_init_value=None,
    dropout_prob=0.0,
)

# 创建完整模型
print('创建 DINO.txt 模型...')
model = DINOTxt(
    model_config=dinotxt_config,
    vision_backbone=vision_backbone,
    text_backbone=text_backbone
)
model.visual_model.backbone = vision_backbone
model.eval()

# 加载 dino.txt 权重（使用内存映射）
print('加载 dino.txt 权重...')
try:
    dinotxt_state_dict = torch.load(DINOTXT_WEIGHTS_PATH, map_location='cpu', weights_only=True, mmap=True)
except TypeError:
    dinotxt_state_dict = torch.load(DINOTXT_WEIGHTS_PATH, map_location='cpu', weights_only=True)

model.load_state_dict(dinotxt_state_dict, strict=False)
print('dino.txt 权重加载完成')

# 删除 state_dict 释放内存
del dinotxt_state_dict
gc.collect()

# 加载 tokenizer
print('加载 Tokenizer...')
with open(BPE_PATH, 'rb') as f:
    bpe_buffer = BytesIO(f.read())
tokenizer = Tokenizer(vocab_path=bpe_buffer)
print('Tokenizer 加载完成')

# 移到设备（CPU）
model = model.to(device)
print(f'模型已加载到 {device}')
print(f'当前内存使用: 请检查系统资源管理器')

## 3. 定义类别名称

In [ ]:
# 城市景观分割类别
CITYSCAPES_CLASSES = (
    'road',
    'sidewalk',
    'building',
    'wall',
    'fence',
    'pole',
    'traffic light',
    'traffic sign',
    'vegetation',
    'terrain',
    'sky',
    'person',
    'rider',
    'car',
    'truck',
    'bus',
    'train',
    'motorcycle',
    'bicycle',
)

# ADE20K 类别 (简化版)
ADE20K_CLASSES = (
    'wall', 'building', 'sky', 'floor', 'tree',
    'ceiling', 'road', 'bed', 'window', 'grass',
    'cabinet', 'sidewalk', 'person', 'ground', 'door',
    'table', 'mountain', 'plant', 'curtain', 'chair',
    'car', 'water', 'painting', 'sofa', 'shelf',
)

# 自定义类别示例
CUSTOM_CLASSES = (
    'person',
    'background',
)

print(f'Cityscapes 类别数: {len(CITYSCAPES_CLASSES)}')
print(f'ADE20K 类别数: {len(ADE20K_CLASSES)}')
print(f'自定义类别数: {len(CUSTOM_CLASSES)}')

## 4. Prompt 模板

In [ ]:
# 参考 CLIP 的 prompt engineering
PROMPT_TEMPLATES = (
    'a photo of a {0}.',
    'a photo of the {0}.',
    'a photo of many {0}.',
    'a bright photo of a {0}.',
    'a dark photo of the {0}.',
    'a photo of a small {0}.',
    'a photo of a large {0}.',
    'a photo of the small {0}.',
    'a photo of the large {0}.',
    'a blurry photo of a {0}.',
    'a blurry photo of the {0}.',
    'a low resolution photo of a {0}.',
    'a low resolution photo of the {0}.',
    'a rendering of a {0}.',
    'a rendering of the {0}.',
    'a cropped photo of a {0}.',
    'a cropped photo of the {0}.',
    'a good photo of a {0}.',
    'a good photo of the {0}.',
    'a close-up photo of a {0}.',
    'a close-up photo of the {0}.',
)

print(f'Prompt 模板数量: {len(PROMPT_TEMPLATES)}')

## 5. 编码函数

In [ ]:
def encode_text_features(class_names, model, tokenizer):
    text_feats = []
    for class_name in class_names:
        text = [template.format(class_name) for template in PROMPT_TEMPLATES]
        tokens = tokenizer.tokenize(text).to(device)
        with torch.no_grad():
            feats = model.encode_text(tokens)
        feats = feats[:, feats.shape[1] // 2 :]
        feats = F.normalize(feats, p=2, dim=-1)
        feats = feats.mean(dim=0)
        feats = F.normalize(feats, p=2, dim=-1)
        text_feats.append(feats)
    return torch.stack(text_feats)


def encode_image_features(model, img):
    B, _, H, W = img.shape
    P = model.visual_model.backbone.patch_size
    new_H = math.ceil(H / P) * P
    new_W = math.ceil(W / P) * P
    if (H, W) != (new_H, new_W):
        img = F.interpolate(img, size=(new_H, new_W), mode='bicubic', align_corners=False)
    B, _, h_i, w_i = img.shape
    with torch.no_grad():
        cls_tokens, _, patch_tokens = model.visual_model.get_class_and_patch_tokens(img)
    blocks_patches = patch_tokens.reshape(B, h_i // P, w_i // P, -1)
    return blocks_patches


class ShortSideResize:
    def __init__(self, size: int):
        self.size = size
    
    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        _, h, w = img.shape
        if w < h:
            new_w = self.size
            new_h = int(self.size * h / w)
        else:
            new_h = self.size
            new_w = int(self.size * w / h)
        return TVTF.resize(img, [new_h, new_w], TVT.InterpolationMode.BICUBIC)


NORMALIZE_IMAGENET = TVT.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

print('编码函数定义完成')

## 6. 预测函数

In [ ]:
def predict_whole(model, img: torch.Tensor, text_features: torch.Tensor):
    _, H, W = img.shape
    img_batch = img.unsqueeze(0)
    blocks_feats = encode_image_features(model, img_batch)
    blocks_feats = blocks_feats.squeeze(0)
    blocks_feats = F.normalize(blocks_feats, p=2, dim=-1)
    cos = torch.einsum('cd,hwd->chw', text_features, blocks_feats)
    cos = F.interpolate(
        cos.unsqueeze(0),
        size=(H, W),
        mode='bilinear',
        align_corners=False
    ).squeeze(0)
    return cos


def predict_slide(model, img: torch.Tensor, text_features: torch.Tensor, side: int = 384, stride: int = 192):
    _, H, W = img.shape
    num_classes = text_features.shape[0]
    probs = torch.zeros([num_classes, H, W], device=device)
    counts = torch.zeros([H, W], device=device)
    h_grids = max(H - side + stride - 1, 0) // stride + 1
    w_grids = max(W - side + stride - 1, 0) // stride + 1
    total_windows = h_grids * w_grids
    print(f'滑动窗口: {h_grids}x{w_grids} = {total_windows} 个窗口')
    for i in range(h_grids):
        for j in range(w_grids):
            y1 = i * stride
            x1 = j * stride
            y2 = min(y1 + side, H)
            x2 = min(x1 + side, W)
            y1 = max(y2 - side, 0)
            x1 = max(x2 - side, 0)
            img_window = img[:, y1:y2, x1:x2]
            with torch.no_grad():
                blocks_feats = encode_image_features(model, img_window.unsqueeze(0))
                blocks_feats = blocks_feats.squeeze(0)
                blocks_feats = F.normalize(blocks_feats, p=2, dim=-1)
                cos = torch.einsum('cd,hwd->chw', text_features, blocks_feats)
                cos = F.interpolate(
                    cos.unsqueeze(0),
                    size=img_window.shape[1:],
                    mode='bilinear',
                    align_corners=False
                ).squeeze(0)
                probs[:, y1:y2, x1:x2] += cos.softmax(dim=0)
                counts[y1:y2, x1:x2] += 1
    probs = probs / counts
    return probs


print('预测函数定义完成')

## 7. 加载测试图像

In [ ]:
def load_image(path: str) -> Image.Image:
    return Image.open(path).convert('RGB')

# 替换为你的图像路径
IMAGE_PATH = r'd:\AI\Git\dinov3.git\notebooks\example.jpg'

img_pil = load_image(IMAGE_PATH)
print(f'图像尺寸: {img_pil.size}')

plt.figure(figsize=(10, 8))
plt.imshow(img_pil)
plt.axis('off')
plt.title('原始图像')
plt.show()

## 8. 图像预处理

In [ ]:
RESIZE_SIZE = 512
MODE = 'whole'
CLASS_NAMES = CITYSCAPES_CLASSES

transform = TVT.Compose([
    TVT.PILToTensor(),
    TVT.ConvertImageDtype(torch.float32),
    ShortSideResize(RESIZE_SIZE),
    NORMALIZE_IMAGENET,
])

img_tensor = transform(img_pil).to(device)
print(f'预处理后尺寸: {img_tensor.shape}')

## 9. 编码文本特征

In [ ]:
print(f'正在编码 {len(CLASS_NAMES)} 个类别...')
text_features = encode_text_features(CLASS_NAMES, model, tokenizer)
text_features = text_features.to(device)
print(f'文本特征形状: {text_features.shape}')

## 10. 执行分割推理

In [ ]:
print(f'执行分割推理 (mode={MODE})...')

if MODE == 'whole':
    pred = predict_whole(model, img_tensor, text_features)
else:
    pred = predict_slide(model, img_tensor, text_features, side=384, stride=192)

pred_labels = pred.argmax(dim=0)

print(f'预测结果形状: {pred.shape}')
print(f'预测标签范围: {pred_labels.min()} ~ {pred_labels.max()}')

## 11. 可视化分割结果

In [ ]:
def get_color_palette(num_classes):
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(num_classes, 3), dtype=np.uint8)
    return colors


colors = get_color_palette(len(CLASS_NAMES))
pred_labels_np = pred_labels.cpu().numpy().astype(np.int32)
H, W = pred_labels_np.shape

seg_color = np.zeros((H, W, 3), dtype=np.uint8)
for label_id in range(len(CLASS_NAMES)):
    seg_color[pred_labels_np == label_id] = colors[label_id]

img_np = np.array(img_pil.resize((W, H)))
overlay = img_np * 0.6 + seg_color * 0.4
overlay = overlay.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_np)
axes[0].set_title('原始图像')
axes[0].axis('off')

axes[1].imshow(seg_color)
axes[1].set_title('分割结果')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title('叠加显示')
axes[2].axis('off')

plt.tight_layout()
plt.show()

unique_labels = np.unique(pred_labels_np)
print('\n检测到的类别:')
print('=' * 40)
for label_id in unique_labels:
    if label_id < len(CLASS_NAMES):
        class_name = CLASS_NAMES[label_id]
        pixel_count = np.sum(pred_labels_np == label_id)
        percentage = pixel_count / pred_labels_np.size * 100
        print(f'  {class_name}: {percentage:.1f}%')

## 12. 使用自定义类别进行分割

In [ ]:
CUSTOM_TEST_CLASSES = (
    'person',
    'animal',
    'vehicle',
    'building',
    'tree',
    'sky',
    'ground',
    'water',
)

print(f'使用自定义类别: {len(CUSTOM_TEST_CLASSES)} 类')

custom_text_features = encode_text_features(CUSTOM_TEST_CLASSES, model, tokenizer)
custom_text_features = custom_text_features.to(device)

print('执行分割...')
with torch.no_grad():
    custom_pred = predict_whole(model, img_tensor, custom_text_features)
    custom_pred_labels = custom_pred.argmax(dim=0)

custom_pred_labels_np = custom_pred_labels.cpu().numpy().astype(np.int32)
H, W = custom_pred_labels_np.shape

# 重新加载/调整原图尺寸（不依赖Step 11的img_np）
img_np = np.array(img_pil.resize((W, H)))

custom_colors = get_color_palette(len(CUSTOM_TEST_CLASSES))
custom_seg_color = np.zeros((H, W, 3), dtype=np.uint8)
for label_id in range(len(CUSTOM_TEST_CLASSES)):
    custom_seg_color[custom_pred_labels_np == label_id] = custom_colors[label_id]

# 确保使用float计算避免溢出
custom_overlay = img_np.astype(np.float32) * 0.6 + custom_seg_color.astype(np.float32) * 0.4
custom_overlay = custom_overlay.astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_np)
axes[0].set_title('原始图像')
axes[0].axis('off')

axes[1].imshow(custom_overlay)
axes[1].set_title('自定义类别分割')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# 打印类别分布
custom_unique = np.unique(custom_pred_labels_np)
print('\n检测到的自定义类别:')
print('=' * 40)
for label_id in custom_unique:
    if label_id < len(CUSTOM_TEST_CLASSES):
        class_name = CUSTOM_TEST_CLASSES[label_id]
        pixel_count = np.sum(custom_pred_labels_np == label_id)
        percentage = pixel_count / custom_pred_labels_np.size * 100
        print(f'  {class_name}: {percentage:.1f}%')

## 总结

### 已完成
1. 加载 DINOv3 + dino.txt 模型 (CPU/GPU 自适应)
2. 支持 Cityscapes、ADE20K 和自定义类别
3. 整图推理 (predict_whole)
4. 滑动窗口推理 (predict_slide) - 适用于大图像
5. 分割结果可视化

### CPU 优化建议
1. 图像尺寸: 建议 resize 到 512 或更小
2. 推理模式: 内存充足用 'whole'，否则用 'slide'
3. 窗口大小: slide 模式下可调整 side 和 stride 参数
4. 类别数量: 自定义类别建议控制在 10 个以内